# 03 - Weather Grid Risk Correlation

## Purpose

Create a day-region risk correlation output between weather conditions and grid incidents.

This notebook aligns weather and grid signals to the same grain before combining them.

## Expected output

`vattenfall_dev.analytics.gold_weather_grid_risk_correlation`

## Main metrics

- average wind speed
- weather alert count
- incident count
- elevated incident count
- total incident duration
- weather risk category
- grid risk category
- weather-grid risk score

## Main idea

This output shows where weather risk and grid incident pressure appear together. It does not prove causality.


In [0]:
from pyspark.sql import functions as F

In [0]:
catalog = "vattenfall_dev"

weather_table = f"{catalog}.analytics.gold_weather_impact_summary"
grid_table = f"{catalog}.analytics.gold_grid_incident_summary"

target_table = f"{catalog}.analytics.gold_weather_grid_risk_correlation"

print("Weather source:", weather_table)
print("Grid source:", grid_table)
print("Target table:", target_table)

In [0]:
weather_df = spark.table(weather_table)
grid_df = spark.table(grid_table)

print("Weather rows:", weather_df.count())
print("Grid incident rows:", grid_df.count())

display(weather_df)
display(grid_df)

In [0]:
grid_day_region_df = (
    grid_df
    .groupBy(
        F.col("event_day").alias("report_day"),
        "region"
    )
    .agg(
        F.sum("incident_count").alias("incident_count"),
        F.sum("elevated_incident_count").alias("elevated_incident_count"),
        F.sum("total_duration_minutes").alias("total_duration_minutes"),
    )
)

display(grid_day_region_df)

In [0]:
weather_grid_df = (
    weather_df.alias("w")
    .join(
        grid_day_region_df.alias("g"),
        on=["report_day", "region"],
        how="left"
    )
    .fillna(
        {
            "incident_count": 0,
            "elevated_incident_count": 0,
            "total_duration_minutes": 0,
        }
    )
    .withColumn(
        "weather_risk_category",
        F.when(F.col("weather_risk_signal") == "WEATHER_RISK", "HIGH")
        .otherwise("LOW")
    )
    .withColumn(
        "grid_risk_category",
        F.when(F.col("elevated_incident_count") >= 3, "HIGH")
        .when(F.col("incident_count") > 0, "MEDIUM")
        .otherwise("LOW")
    )
    .withColumn(
        "weather_grid_risk_score",
        F.when(
            (F.col("weather_risk_category") == "HIGH")
            & (F.col("grid_risk_category") == "HIGH"),
            2
        )
        .when(
            (F.col("weather_risk_category") == "HIGH")
            | (F.col("grid_risk_category").isin("HIGH", "MEDIUM")),
            1
        )
        .otherwise(0)
    )
)

display(weather_grid_df)

In [0]:
(
    weather_grid_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Written Night Shift weather-grid risk correlation table:", target_table)
print("Rows written:", weather_grid_df.count())

In [0]:
result_df = spark.table(target_table)

print("Rows in weather-grid risk correlation:", result_df.count())
print("Columns:", result_df.columns)

display(result_df)

print("Weather-grid risk score distribution:")
result_df.groupBy("weather_grid_risk_score").count().show()

print("Weather risk category distribution:")
result_df.groupBy("weather_risk_category").count().show()

print("Grid risk category distribution:")
result_df.groupBy("grid_risk_category").count().show()

In [0]:
duplicate_rows = (
    result_df
    .groupBy("report_day", "region")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate report_day-region rows:", duplicate_rows)

if duplicate_rows > 0:
    raise ValueError("Weather-grid risk validation failed: duplicate report_day-region rows found.")

print("Weather-grid risk correlation validation passed.")